# Constrained LQR CUDA adversarial stress validation

Start a fresh Kaggle notebook with a CUDA GPU and Internet access. This is the correctness and sanitizer workflow, not a timing benchmark. It records the complete machine and toolchain description; runs the standard and extended deterministic CPU suites; executes the shared corpus through CUDA kernel emulation and native CUDA; and runs Compute Sanitizer memcheck, initcheck, racecheck, and synccheck over both the focused native suite and the extended adversarial suite. The fixed corpus includes infeasibility, redundant constraints, zero and changing dimensions, non-power-of-two and long horizons, scaling, nonunique multipliers, reduced-Hessian failures, and workspace reuse.

In [ ]:
%%bash
set -euo pipefail

repo=/kaggle/working/constrained_lqr_elimination
revision="${CLQR_REVISION:-main}"

if [[ ! -d "${repo}/.git" ]]; then
  git clone --filter=blob:none --no-checkout \
    https://github.com/joaospinto/constrained_lqr_elimination.git \
    "${repo}"
fi
git -C "${repo}" fetch --depth 1 origin "${revision}"
git -C "${repo}" checkout --detach FETCH_HEAD
git -C "${repo}" rev-parse HEAD

cd "${repo}"
CLQR_PRECISIONS="FP64 FP32" \
CLQR_CUDA_ARCH="${CLQR_CUDA_ARCH:-60}" \
CLQR_SANITIZER_TOOLS="memcheck initcheck racecheck synccheck" \
bash scripts/notebook_cuda_stress.sh 2>&1 | \
  tee /kaggle/working/clqr_cuda_stress_report.txt


The complete machine description and validation transcript are saved as `/kaggle/working/clqr_cuda_stress_report.txt`. The cell defaults to `main`; set `CLQR_REVISION` to validate another branch or exact commit. Set `CLQR_PRECISIONS="FP64 FP32"` in the cell when both native precisions are desired; FP64 alone is the faster default for sanitizer-heavy runs.